# **Setup**

## **Settings**

In [ ]:
# Everything you might change lives here.
BRANCH = "main"          # repo branch to clone ("main" once a step is merged)
SEED = 42
EPOCHS = 8
OUT = "/kaggle/working/baseline"   # kept as notebook output

# Which parts run. For a saved full run: checks and smoke off, full on.
RUN_CHECKS = False       # debugging checks 2 and 3 (about 1 min)
RUN_SMOKE = False        # smoke test: 3 epochs on 10,000 scans (about 3 min)
RUN_FULL = True          # full run: EPOCHS on all 97,479 scans (about 35 min)

## **Clone the code**

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO = Path("/tmp/oct-calibration")
if not (REPO / "src" / "train.py").exists():
    subprocess.run(["rm", "-rf", str(REPO)])
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/zyanislam/oct-calibration.git", str(REPO)],
                       capture_output=True, text=True)
    assert r.returncode == 0, f"git clone failed. Is Internet on in Settings?\n{r.stderr}"

# Python remembers "no module here" for a path added before the folder existed: clear it
sys.path_importer_cache.pop(str(REPO), None)
importlib.invalidate_caches()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

commit = subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print("code:", BRANCH, "@", commit)

## **Copy the data to local disk**

In [ ]:
import shutil, time
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from tqdm.auto import tqdm

SOURCE = Path("/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3-256")
ROOT = Path("/tmp/oct256")   # local disk: fast, wiped when the session ends
assert SOURCE.is_dir(), f"dataset not found at {SOURCE}: attach it under Input"

paths = pd.read_csv(REPO / "splits/oct_split_fold0.csv")["path"]

def copy_one(rel):
    dst = ROOT / rel
    if not dst.exists():                       # rerunning skips files already copied
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(SOURCE / rel, dst)

t = time.time()
with ThreadPoolExecutor(max_workers=64) as pool:
    list(tqdm(pool.map(copy_one, paths), total=len(paths), desc="Copying"))
print(f"{len(paths)} scans on local disk in {time.time() - t:.0f}s")

## **Load the code, GPU and data**

In [ ]:
import torch
from src.dataset import make_loaders
from src.model import build_model
from src.train import set_seed, class_weights, first_loss, overfit_one_batch, subset_loader, fit

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 x1 in Settings"
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0), "| seed", SEED)

set_seed(SEED)
loaders = make_loaders(ROOT, batch_size=64, num_workers=4)
weights = class_weights(loaders["train"].dataset.labels)
print({k: len(v.dataset) for k, v in loaders.items()})
print("class weights CNV/DME/DRUSEN/NORMAL:", [round(w, 2) for w in weights.tolist()])

# **Baseline**

## **Checks 2 and 3**

In [ ]:
if RUN_CHECKS:
    set_seed(SEED)
    model = build_model().to(device)
    print(f"Check 2 · first loss: {first_loss(model, loaders['train'], device):.3f}  (expect about 1.4–1.8)")
    losses = overfit_one_batch(model, loaders["train"], device, steps=50)
    print("Check 3 · overfit one batch:", [round(l, 3) for l in losses[::10]], "→ final", round(losses[-1], 4))
else:
    print("skipped (RUN_CHECKS = False)")

## **Smoke test**

In [ ]:
if RUN_SMOKE:
    set_seed(SEED)
    smoke = dict(loaders, train=subset_loader(loaders["train"], 10_000, seed=SEED))
    fit(build_model(), smoke, "/kaggle/working/smoke", epochs=3, weights=weights, device=device)
else:
    print("skipped (RUN_SMOKE = False)")

## **Full run**

In [ ]:
if RUN_FULL:
    set_seed(SEED)
    fit(build_model(), loaders, OUT, epochs=EPOCHS, weights=weights, device=device)
else:
    print("skipped (RUN_FULL = False)")

## **Results**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from src.data import CLASSES

log_path = Path(OUT) / "log.csv"
if log_path.exists():
    log = pd.read_csv(log_path)
    display(log)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(log.epoch, log.train_loss, "o--", label="train loss")
    ax[0].plot(log.epoch, log.val_loss, "o-", label="val loss")
    ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(log.epoch, log.val_macro_f1, "o-", label="val macro-F1")
    ax[1].plot(log.epoch, log.val_accuracy, "s-", label="val accuracy")
    ax[1].axhline(0.90, ls=":", color="red", label="target 0.90")
    ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.show()

    for split in ("val", "test"):
        z = torch.load(Path(OUT) / f"logits_{split}.pt")
        y, p = z["labels"].numpy(), z["logits"].argmax(1).numpy()
        print(f"\n=== best.pt on {split} ===")
        print(classification_report(y, p, target_names=CLASSES, digits=3))
        print(pd.DataFrame(confusion_matrix(y, p), index=CLASSES, columns=CLASSES))
else:
    print("no results yet: run the full run first")